In [1]:
# Most code copied from Reference 1
!python -m pip install numpy
!python -m pip install matplotlib
!python -m pip install pandas
!python -m pip install scikit-learn

In [2]:
# Load necessary modules and libraries, reference [1]

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import learning_curve
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic, Matern, ExpSineSquared,DotProduct

import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
import io
from google.colab import files
uploaded = files.upload() #"Surface_features.csv"

### After running this section. press choose file and select Surface_features.csv ###

Saving Surface_features.csv to Surface_features.csv


In [4]:
# Load the data, reference [1]


Geometry1 = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0, usecols=(4,8,9,10,11,12,14)).iloc[0:45]
Geometry = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0, usecols=(4,6,7,8,9,10,11,12)).iloc[0:45].values

Ra_ch = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(5,)).iloc[0:45].values
Ra_ch = Ra_ch[:,0]
ks = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(13,)).iloc[0:45].values
ks = ks[:,0]



Geometry1["ks"]= np.divide(ks,Ra_ch)
Geometry1["krms_ch"]= np.divide(Geometry1["krms_ch"],Ra_ch)


Geometry1.rename({'krms_ch': '$k_{rms}/R_a$',
                  'pro_ch': '$P_o$',
                  'ESx_ch': '$E_x$',
                  'ESz_ch': '$E_z$',
                  'sk_ch': '$S_k$',
                  'ku_ch': '$K_u$',
                  'ks': '$k_s/R_a$',
                  'label': 'Label',
                  }, axis='columns', errors="raise",inplace = True)


In [5]:
# Data reconfiguration, to be used in ML, reference [1]

X = Geometry
y = np.divide(ks,Ra_ch)


X[:,0] = np.divide(X[:,0],Ra_ch)
X[:,2] = np.abs(X[:,2])


# Generate secondary features and append them to the original dataset

n,m = X.shape
X0 = np.ones((n,1))
X1 = np.ones((n,1))
X2 = np.ones((n,1))
X3 = np.ones((n,1))
X4 = np.ones((n,1))
X5 = np.ones((n,1))
X6 = np.ones((n,1))
X7 = np.ones((n,1))
X8 = np.ones((n,1))
X9 = np.ones((n,1))




X1[:,0] = np.transpose(X[:,4]*X[:,5])
X2[:,0] = np.transpose(X[:,4]*X[:,6])
X3[:,0] = np.transpose(X[:,4]*X[:,7])
X4[:,0] = np.transpose(X[:,5]*X[:,6])
X5[:,0] = np.transpose(X[:,5]*X[:,7])
X6[:,0] = np.transpose(X[:,6]*X[:,7])
X7[:,0] = np.transpose(X[:,4]*X[:,4])
X8[:,0] = np.transpose(X[:,5]*X[:,5])
X9[:,0] = np.transpose(X[:,6]*X[:,6])




X = np.hstack((X,X1))
X = np.hstack((X,X2))
X = np.hstack((X,X3))
X = np.hstack((X,X4))
X = np.hstack((X,X5))
X = np.hstack((X,X6))
X = np.hstack((X,X7))
X = np.hstack((X,X8))
X = np.hstack((X,X9))




In [6]:
# Best linear estimation, reference [1]


reg = LinearRegression().fit(X, y)
reg.score(X, y)
yn=reg.predict(X)
print("Mean err: %f" % np.mean(100.*abs(yn-y)/(y)))
print("Max err: %f" % max(100.*abs(yn-y)/(y)))





Mean err: 14.945903
Max err: 51.586233


In [7]:
# Define three files that store the best ML prediction based on either L_\infty, L_1 and a custom L norms, reference [1]

filename1 = 'MLP_Linf.sav'
filename2 = 'MLP_L1.sav'
filename3 = 'MLP_L.sav'

In [8]:
Layers = np.array([5,4,4])
N_of_Param = 17*Layers[0]+Layers[0]+Layers[0]*Layers[1]+Layers[1]+Layers[1]*Layers[2]+Layers[2]+Layers[2]*1+1
print("Number of parameters: %f" % N_of_Param)

Number of parameters: 139.000000


In [12]:
# Perform ML training --- it may take some time, reference [1]
# Adjust ranges for by1 through by4 for faster (but potentially less accurate) results.



miny1=100
miny2=50
miny3=1
for by1 in [5]: #18
    for by2 in [4]: #7
        for by3 in [4]: #7

            my_mlp_obj_1 = MLPRegressor(hidden_layer_sizes=(by1,by2,by3), max_iter=200000, alpha=1,
                    solver='lbfgs', verbose=0, tol=1e-5, activation = 'relu', learning_rate_init=.01)
            by4=0.
            print("by1: %f" % by1)
            print("by2: %f" % by2)
            print("by3: %f" % by3)

            while by4<100.:
                by4=by4+1
                print("Outer Iteration: %f" % by4)

                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
                my_mlp_obj_1.fit(X_train, y_train)
                yn=my_mlp_obj_1.predict(X)


                if miny1>max(100.*abs(yn-y)/y):
                    pickle.dump(my_mlp_obj_1, open(filename1, 'wb'))
                    miny1=max(100.*abs(yn-y)/y)
                    print("Miny1: %f" % miny1)

                if miny2>np.mean(100.*abs(yn-y)/y):
                    pickle.dump(my_mlp_obj_1, open(filename2, 'wb'))
                    miny2=np.mean(100.*abs(yn-y)/y)
                    print("Miny2: %f" % miny2)

                if miny3>max(abs(np.log(yn/y))):
                    pickle.dump(my_mlp_obj_1, open(filename3, 'wb'))
                    miny3=max(abs(np.log(yn/y)))
                    print("Miny3: %f" % miny3)




print("by1: %f" % by1)
print("by2: %f" % by2)
print("by3: %f" % by3)



by1: 5.000000
by2: 4.000000
by3: 4.000000
Outer Iteration: 1.000000
Miny1: 93.007290
Miny2: 10.097451
Miny3: 0.657558
Outer Iteration: 2.000000
Outer Iteration: 3.000000
Outer Iteration: 4.000000
Miny1: 42.616023
Miny3: 0.514342
Outer Iteration: 5.000000
Outer Iteration: 6.000000
Outer Iteration: 7.000000
Outer Iteration: 8.000000
Outer Iteration: 9.000000
Outer Iteration: 10.000000
Outer Iteration: 11.000000
Miny3: 0.507427
Outer Iteration: 12.000000
Outer Iteration: 13.000000
Outer Iteration: 14.000000
Outer Iteration: 15.000000
Outer Iteration: 16.000000
Outer Iteration: 17.000000
Miny3: 0.481378
Outer Iteration: 18.000000
Outer Iteration: 19.000000
Miny3: 0.376594
Outer Iteration: 20.000000
Outer Iteration: 21.000000
Outer Iteration: 22.000000
Outer Iteration: 23.000000
Outer Iteration: 24.000000
Miny2: 8.947048
Outer Iteration: 25.000000
Outer Iteration: 26.000000
Outer Iteration: 27.000000
Outer Iteration: 28.000000
Outer Iteration: 29.000000
Miny1: 40.487898
Outer Iteration: 30.

In [13]:
# Load either file1, file2 or file3 to extract the results, reference [1]

loaded_model = pickle.load(open(filename2, 'rb'))
loaded_model.get_params()


{'activation': 'relu',
 'alpha': 1,
 'batch_size': 'auto',
 'beta_1': 0.9,
 'beta_2': 0.999,
 'early_stopping': False,
 'epsilon': 1e-08,
 'hidden_layer_sizes': (5, 4, 4),
 'learning_rate': 'constant',
 'learning_rate_init': 0.01,
 'max_fun': 15000,
 'max_iter': 200000,
 'momentum': 0.9,
 'n_iter_no_change': 10,
 'nesterovs_momentum': True,
 'power_t': 0.5,
 'random_state': None,
 'shuffle': True,
 'solver': 'lbfgs',
 'tol': 1e-05,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [14]:
#reference [1]
yn = loaded_model.predict(X)
print("PREDICTED k_s/R_a= ")
print(yn)
print("Max err: %f" % max(100.*abs(yn-y)/(y)))
print("mean err: %f" % np.mean(100.*abs(yn-y)/(y)))


Error=pd.DataFrame()
Error["$k_s/Ra$"]= y
Error["$k_{sp}/Ra$"]= yn
Error["$error$"]= (100.*(yn-y)/(y))
Error["Label"]= Geometry1["Label"]

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(Error)

PREDICTED k_s/R_a= 
[ 3.33304532  4.18523853  3.14916732  7.50660363 12.31306515  7.68967192
 10.98922716 14.99620805 10.00847448  7.44540847 12.45291233  3.14090175
  9.22580727 12.57847952  9.14905216  8.00029327  8.5704002  10.41953317
  8.5213709  11.46803766  3.63504723  3.63811778  3.60227314  3.42318834
  5.35841638  7.1131451   3.03797625  6.84493387  1.7073067  14.95753495
  3.22162066  3.07415181  4.03476877  4.44838052  3.39735169  3.41291563
  1.95532212  1.79165112  1.95333216  2.60952586  2.61829044  2.62079276
  4.11780637  5.91973263  2.83181414]
Max err: 58.655641
mean err: 8.947048
     $k_s/Ra$  $k_{sp}/Ra$    $error$ Label
0    3.319588     3.333045   0.405403   DNS
1    4.174497     4.185239   0.257322   DNS
2    3.051813     3.149167   3.190033   DNS
3    7.168421     7.506604   4.717672   DNS
4   12.688976    12.313065  -2.962502   DNS
5    7.243094     7.689672   6.165570   DNS
6   10.628458    10.989227   3.394365   DNS
7   14.615804    14.996208   2.602691   D